# Bronze Layer: Detailed Extraction Notebook

**Target Tables:**
- **Read:** `bronze_ad_links` (`bronze.GeneralSearch`)
- **Write:** `bronze_extraction_data` (`bronze.InformationExtraction`)

**Objective:**
Extracts detailed product specifications, titles, descriptions, and listing prices for discovered links, saving raw extracted JSON/dict specs into the `bronze_extraction_data` table.

## 1. Setup and Imports
Import system libraries, SQLAlchemy, Polars, and application service & model modules.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is available in system path
project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

import polars as pl
from sqlalchemy import insert, select, update
from sqlalchemy.orm import Session

from app.config import db_engine
from app.models import bronze

# Importing from 'app' module
from app.services import OLXCrawler
from app.utils import read_search_locations_metadata

## 2. Initialize Extraction Buffers
Set up containers for extracted listing details and sold/unavailable item status updates.

In [2]:
# Container for extracted listing detail dictionaries
extraction_data_list = []

# Container for sold or unavailable listing updates
not_available_products = []

## 3. Query Pending Links for Extraction
Fetch discovered links from `bronze.GeneralSearch` filtered by store and target region abbreviations.

In [3]:
# Load region metadata
brazil_data_location = (
    read_search_locations_metadata()
    .get('search_locations', {})
    .get('brazil', {})
)
regions = brazil_data_location.get('regions', [])
abbreviation_list = [region.get('abreviation') for region in regions]

# Query available general search links from database
with db_engine.connect() as connection:
    df_olx = pl.read_database(
        select(bronze.GeneralSearch).where(
            (bronze.GeneralSearch.region.in_(abbreviation_list))
            & (bronze.GeneralSearch.store.is_('OLX'))
        ),
        connection=connection
    )

## 4. Run Detailed Page Extractor
Scrape product specifications for the listing batch.

In [4]:
if not df_olx.is_empty():
    df_olx_batch = df_olx.head(100)
    olx_crawler = OLXCrawler(base_data_list=extraction_data_list)
    await olx_crawler.scrap_specific_information(
        df=df_olx_batch,
        not_available_products=not_available_products
    )

## 5. Persist Results to Bronze Extraction Table (`bronze_extraction_data`)
Insert detailed listing records into `bronze.InformationExtraction` and update listing availability status.

In [5]:
extraction_data_df = pl.DataFrame(extraction_data_list)
updated_df = pl.DataFrame(not_available_products)

if not extraction_data_df.is_empty():
    with Session(db_engine) as session:
        session.execute(insert(bronze.InformationExtraction), extraction_data_df.to_dicts())
        if not updated_df.is_empty():
            session.execute(update(bronze.GeneralSearch), updated_df.to_dicts())
        session.commit()
        print(f"Successfully saved {len(extraction_data_df)} extracted product details.")
else:
    print("No extraction data found to insert.")